In [1]:
import numpy as np

def max_pool2d_forward(x: np.ndarray, kernel_size, stride, padding):
    """
    :param x: 输入特征图 [N, C, H_in, W_in]
    :param kernel_size: 池化核 (kh, kw)
    :param stride: 步幅 (sh, sw)
    :param padding: 上下左右填充 (ph, pw)
    :return: pool_out [N, C, H_out, W_out]
    """
    N, C, H_in, W_in = x.shape
    kh, kw = kernel_size
    sh, sw = stride
    ph, pw = padding

    # 1. 边缘填充：四周补0
    x_pad = np.pad(x, pad_width=((0,0),(0,0),(ph,ph),(pw,pw)), mode='constant', constant_values=0)
    H_pad = H_in + 2 * ph
    W_pad = W_in + 2 * pw

    # 2. 计算输出尺寸
    H_out = (H_pad - kh) // sh + 1
    W_out = (W_pad - kw) // sw + 1

    # 3. 初始化输出
    out = np.zeros((N, C, H_out, W_out))

    # 4. 滑动窗口遍历取最大值
    for i in range(H_out):
        h_start = i * sh
        h_end = h_start + kh
        for j in range(W_out):
            w_start = j * sw
            w_end = w_start + kw
            # 截取池化窗口
            window = x_pad[:, :, h_start:h_end, w_start:w_end]
            out[:, :, i, j] = np.max(window, axis=(-2,-1))
    return out

# ----------------测试用例----------------
if __name__ == "__main__":
    # 构造输入：[batch=2, channel=3, H=8, W=8]
    inp = np.random.randn(2,3,8,8)
    # 超参：池化3×3, stride=2, padding=1
    pool_out = max_pool2d_forward(inp, kernel_size=(3,3), stride=(2,2), padding=(1,1))
    print("输入shape:", inp.shape)
    print("池化输出shape:", pool_out.shape)

输入shape: (2, 3, 8, 8)
池化输出shape: (2, 3, 4, 4)


In [1]:
import torch
import torch.nn as nn

def my_max_pool2d(x:torch.Tensor, kernel_size, stride, padding):
    N,C,H_in,W_in = x.shape
    kh,kw = kernel_size
    sh,sw = stride
    ph,pw = padding

    # 填充
    x_pad = torch.nn.functional.pad(x, [pw,pw,ph,ph], value=0.)
    H_pad = H_in + 2*ph
    W_pad = W_in + 2*pw

    H_out = (H_pad - kh)//sh + 1
    W_out = (W_pad - kw)//sw + 1
    out = torch.empty(N,C,H_out,W_out)

    for i in range(H_out):
        hs = i*sh
        he = hs + kh
        for j in range(W_out):
            ws = j*sw
            we = ws + kw
            window = x_pad[:,:,hs:he,ws:we]
            out[:,:,i,j] = torch.amax(window, dim=(-2,-1))
    return out

# 测试
if __name__ == "__main__":
    x = torch.randn(1,1,6,6)
    res = my_max_pool2d(x, kernel_size=(2,2), stride=(2,2), padding=(0,0))
    print(res.shape)

torch.Size([1, 1, 3, 3])


In [2]:


class NiN_Block(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super().__init__()
        # 使用Sequential搭建NiN模块：1个普通卷积 + 2层1×1卷积，每层后接ReLU
        self.block = nn.Sequential(
            # 第一层：常规卷积
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
            nn.ReLU(inplace=True),
            # 第二层：1×1卷积
            nn.Conv2d(out_channels, out_channels, kernel_size=1, stride=1, padding=0),
            nn.ReLU(inplace=True),
            # 第三层：1×1卷积
            nn.Conv2d(out_channels, out_channels, kernel_size=1, stride=1, padding=0),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.block(x)

# ----------------测试示例----------------
if __name__ == "__main__":
    # 实例化NiN块：输入3通道，输出96通道，卷积核11×11，stride=4，padding=0（参考原版NiN首层）
    nin_layer = NiN_Block(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=0)
    # 构造随机输入 [N,C,H,W] = [2,3,224,224]
    x = torch.randn(2, 3, 224, 224)
    out = nin_layer(x)
    print("输入尺寸:", x.shape)
    print("NiN块输出尺寸:", out.shape)

输入尺寸: torch.Size([2, 3, 224, 224])
NiN块输出尺寸: torch.Size([2, 96, 54, 54])


In [3]:


class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super().__init__()
        # 主干分支：2层3×3卷积 + BN
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, stride=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # 短路分支：use_1x1conv=True时启用1×1卷积做维度匹配
        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.shortcut = nn.Identity()  # 恒等映射，直接传输入

    def forward(self, x):
        # 主干前向
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        # 残差相加 f(x)+x
        out = out + self.shortcut(x)
        return torch.relu(out)

# ----------------测试代码----------------
if __name__ == "__main__":
    # 场景1：通道/尺寸一致，不用1×1卷积
    blk1 = Residual(in_channels=64, out_channels=64, stride=1, use_1x1conv=False)
    x1 = torch.randn(2,64,32,32)
    print("无1×1conv输出shape：", blk1(x1).shape)

    # 场景2：通道/尺寸变化，开启1×1卷积对齐维度
    blk2 = Residual(in_channels=64, out_channels=128, stride=2, use_1x1conv=True)
    x2 = torch.randn(2,64,32,32)
    print("启用1×1conv输出shape：", blk2(x2).shape)

无1×1conv输出shape： torch.Size([2, 64, 32, 32])
启用1×1conv输出shape： torch.Size([2, 128, 16, 16])


In [4]:
from torchvision import transforms

# 构建图像增强流水线
train_transform = transforms.Compose([
    # 1. 随机裁剪：面积占比0.08~1，再缩放到224×224
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    # 2. 50%概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 随机亮度、对比度、饱和度变化，变化系数0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.0),
    # 4. 转为PyTorch张量
    transforms.ToTensor()
])

In [7]:
from torchvision import datasets
# 加载内置数据集，不用本地图片
dataset = datasets.FashionMNIST(root="./data",train=True,transform=train_transform,download=True)
img_tensor,label = dataset[0]
print(img_tensor.shape)

torch.Size([1, 224, 224])


In [6]:
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, num_classes, eps=0.1):
    """
    :param logits: 模型原始输出 [N, K]，未经过softmax
    :param labels: 真实标签索引 [N]，数值0~K-1
    :param num_classes: 分类总数K
    :param eps: 平滑系数ε，题目默认0.1
    :return: 批量平均损失标量
    """
    # 1. 计算softmax概率
    prob = F.softmax(logits, dim=-1)
    log_prob = torch.log(prob + 1e-8)  # +极小值防log(0)
    
    # 2. 构造平滑后的标签
    smooth_target = torch.full_like(log_prob, fill_value=eps / (num_classes - 1))
    # 真实类别位置赋值1-eps
    smooth_target.scatter_(dim=1, index=labels.unsqueeze(1), value=1. - eps)
    
    # 3. 计算交叉熵损失
    loss = -torch.sum(smooth_target * log_prob, dim=1)
    return torch.mean(loss)

# ----------------测试示例----------------
if __name__ == "__main__":
    # 5样本，10分类
    batch_logits = torch.randn(5, 10)
    batch_label = torch.tensor([1,3,0,7,5])
    loss_val = label_smoothing_cross_entropy(batch_logits, batch_label, num_classes=10, eps=0.1)
    print("标签平滑损失值：", loss_val.item())

标签平滑损失值： 2.590597629547119
